# Week 01 RAG Foundations

RAG를 처음 공부하는 사람을 위한 1주차 정리 노트입니다.

## 이 노트의 목표

- RAG와 파인튜닝의 차이를 명확하게 이해한다.
- 기업 환경에서 왜 VNET, 온프레미스, 콘텐츠 필터링이 함께 이야기되는지 이해한다.
- RAG 파이프라인과 데이터 플로우를 단계별로 설명할 수 있다.
- 청킹이 retrieval 품질에 어떤 영향을 주는지 감을 잡는다.
- 2주차 이후의 실습을 위한 공통 용어를 정리한다.


## 어떻게 읽으면 좋은가

1. 먼저 `RAG vs 파인튜닝` 구간을 읽고 큰 그림을 잡습니다.
2. 그다음 `RAG 파이프라인`과 `데이터 플로우`를 봅니다.
3. 기업 보안 관점의 RAG를 이해한 뒤, `청킹`으로 넘어갑니다.
4. 마지막에 있는 체크리스트와 질문으로 스스로 설명해 봅니다.

이 노트는 코드보다 **개념을 탄탄하게 정리하는 것**에 초점을 둡니다.


## 1주차 핵심 키워드

- `RAG`
- `Fine-tuning`
- `Retriever`
- `Embedding`
- `Vector Store`
- `Chunk`
- `Chunk Overlap`
- `Grounding`
- `Hallucination`
- `VNET`
- `On-Premises`
- `Content Filtering`
- `Access Control`
- `Metadata`
- `Chunk Boundary`


## 왜 기업에서는 RAG를 보안과 함께 보는가

개인 공부용 RAG는 보통 "문서를 넣고 검색해서 답변 받는 시스템"으로 이해하면 충분합니다. 하지만 회사 내부에서 RAG를 쓰는 순간, 단순히 정확한 답변만으로는 부족합니다.

기업 환경에서는 보통 다음 질문이 먼저 나옵니다.

- 사내 문서를 외부로 보내도 되는가
- 누가 어떤 문서에 접근할 수 있는가
- 입력과 출력에 민감 정보가 섞였는지 어떻게 통제할 것인가
- 모델이 사내 정책과 다른 답을 했을 때 어떻게 막을 것인가

즉, 기업용 RAG는 단순한 검색 시스템이 아니라 아래 요소들이 함께 붙는 형태에 가깝습니다.

```text
사용자 -> 애플리케이션 -> 입력 필터링 -> 검색기 -> 사내 문서 저장소
                               -> 프롬프트 구성 -> LLM -> 출력 필터링 -> 사용자
```

여기서 중요한 포인트는, **모델 자체보다 데이터의 흐름과 통제 지점**이 더 중요해진다는 점입니다.


## VNET, 온프레미스, 콘텐츠 필터링 간단 정리

### 1. VNET

VNET은 보통 모델 호출이나 데이터 접근 경로를 사설 네트워크 안쪽에 묶어, 외부 인터넷 노출을 줄이기 위한 네트워크 설계로 이해하면 됩니다.

- 장점: 외부 노출 감소, 내부 자원 연결 용이, 보안 정책 적용이 쉬움
- RAG 관점: 문서 저장소, 벡터 DB, 애플리케이션, 모델 연결 경로를 더 통제하기 좋음

### 2. 온프레미스

온프레미스는 서버와 데이터가 회사 내부 인프라 안에서 운영되는 형태입니다.

- 장점: 민감 데이터 통제력 높음, 규제 대응에 유리할 수 있음
- 단점: 운영 비용과 관리 책임이 커짐
- RAG 관점: 원문 문서, 인덱스, 로그, 접근 권한까지 내부에서 직접 관리 가능

### 3. 콘텐츠 필터링

콘텐츠 필터링은 입력과 출력을 정책 기준에 맞게 검사하고 차단하거나 수정하는 레이어입니다.

- 입력 필터링: 민감 정보, 금지된 프롬프트, 악성 요청 차단
- 출력 필터링: 유해 답변, 정책 위반 답변, 개인정보 노출 차단
- RAG 관점: 문서를 잘 찾는 것만큼이나, **찾은 정보를 안전하게 사용하는 것**이 중요함


## RAG란 무엇인가

RAG는 `Retrieval-Augmented Generation`의 약자입니다.

핵심 아이디어는 단순합니다.

1. 질문과 관련된 문서를 먼저 검색합니다.
2. 검색된 문서를 프롬프트에 함께 넣습니다.
3. 모델은 그 문서를 근거로 답변합니다.

즉, RAG는 모델의 뇌를 다시 훈련시키는 방식이 아니라, **필요한 순간에 외부 지식을 옆에 붙여 주는 방식**입니다.

그래서 RAG는 특히 아래 상황에서 강합니다.

- 최신 정보가 자주 바뀌는 경우
- 사내 문서처럼 외부 모델이 원래 모르는 정보를 써야 하는 경우
- 답변의 근거 문서를 함께 보여 주고 싶은 경우


## 파인튜닝이란 무엇인가

파인튜닝은 모델의 가중치나 행동 특성을 특정 태스크에 더 잘 맞도록 조정하는 방식입니다.

예를 들면 다음과 같은 목적에 잘 맞습니다.

- 특정한 답변 스타일을 안정적으로 유지하고 싶을 때
- 특정 포맷의 출력이 계속 필요할 때
- 분류, 추출, 요약 같은 반복 태스크 성능을 올리고 싶을 때
- 특정 도메인 표현 방식에 모델을 익숙하게 만들고 싶을 때

중요한 점은, 파인튜닝은 보통 **모델의 행동을 바꾸는 것**에 가깝고, RAG는 **모델이 참고하는 지식을 바꾸는 것**에 가깝다는 점입니다.


## RAG와 파인튜닝의 차이점

| 항목 | RAG | 파인튜닝 |
|---|---|---|
| 핵심 목적 | 외부 지식을 검색해 답변 근거를 제공 | 모델 행동과 출력 성향을 조정 |
| 지식의 위치 | 문서 저장소, 벡터 DB, 검색 인프라 | 모델 가중치 또는 학습 결과 |
| 최신 정보 반영 | 빠름 | 느림 |
| 운영 형태 | 문서 업데이트 중심 | 데이터셋 준비와 재학습 중심 |
| 근거 제시 | 비교적 쉬움 | 상대적으로 어려움 |
| 문서 보안 제어 | 아키텍처 설계에 따라 세밀하게 가능 | 학습 데이터 관리가 더 중요 |
| 비용 구조 | 검색 인프라 + 추론 비용 | 학습 비용 + 추론 비용 |
| 잘 맞는 문제 | 사내 문서 QA, 정책 검색, 최신 자료 기반 응답 | 스타일 고정, 형식화된 출력, 태스크 특화 |

정리하면:

- **RAG는 지식 주입에 강함**
- **파인튜닝은 행동 조정에 강함**

실무에서는 둘을 같이 쓰는 경우도 많습니다.

- RAG로 최신 사내 지식을 붙이고
- 파인튜닝으로 답변 형식과 말투, 분류 습관을 안정화하는 식입니다.


## 언제 RAG를 쓰고, 언제 파인튜닝을 쓸까

### RAG가 더 적합한 경우

- 문서가 자주 바뀐다.
- 사내 정책, 매뉴얼, 위키 같은 비공개 지식을 써야 한다.
- 답변 근거를 함께 보여 주고 싶다.
- 잘못된 답변이 나오면 어떤 문서를 봤는지 추적해야 한다.

### 파인튜닝이 더 적합한 경우

- 답변 형식을 엄격하게 맞춰야 한다.
- 특정 태스크를 반복적으로 수행해야 한다.
- 도메인 표현 방식과 말투를 안정화하고 싶다.

### 둘을 같이 쓰면 좋은 경우

- 최신 문서 근거는 RAG로 가져오고
- 최종 출력 포맷과 업무 스타일은 파인튜닝으로 고정하고 싶을 때


## RAG 파이프라인 큰 그림

RAG는 보통 두 단계로 나눠 보면 이해가 쉽습니다.

### 1. 적재 단계(Indexing / Ingestion)

```text
문서 수집 -> 정제 -> 청킹 -> 임베딩 생성 -> 벡터 저장소 적재 -> 메타데이터 저장
```

### 2. 질의 단계(Retrieval / Generation)

```text
사용자 질문 -> 질문 임베딩 -> 관련 청크 검색 -> 프롬프트 구성 -> LLM 답변 생성
```

이 흐름을 보면 RAG의 핵심은 두 가지입니다.

- 문서를 **어떻게 저장하느냐**
- 질문이 들어왔을 때 **어떻게 잘 찾느냐**


## RAG 데이터 플로우 자세히 보기

### 문서 적재 플로우

1. 원본 문서를 수집합니다.
2. PDF, HTML, DOCX, 위키 문서 등에서 텍스트를 추출합니다.
3. 불필요한 공백, 깨진 문자, 반복 헤더/푸터를 정리합니다.
4. 문서를 청크 단위로 나눕니다.
5. 각 청크를 임베딩 벡터로 변환합니다.
6. 벡터와 메타데이터를 벡터 저장소에 넣습니다.

### 질의 응답 플로우

1. 사용자가 질문합니다.
2. 질문을 임베딩 벡터로 바꿉니다.
3. 벡터 저장소에서 비슷한 청크를 찾습니다.
4. 찾은 청크를 프롬프트에 붙입니다.
5. 모델이 그 근거를 바탕으로 답변합니다.
6. 필요하면 출처, 링크, 문서 제목을 함께 보여 줍니다.

여기서 RAG 품질을 결정하는 포인트는 다음과 같습니다.

- 문서 정제가 잘 되었는가
- 청킹이 자연스러운가
- 임베딩이 문서 의미를 잘 담는가
- 검색 결과가 질문과 충분히 관련 있는가
- 프롬프트가 검색 결과를 제대로 활용하는가


## 회사 내부에서 보안을 위해 사용하는 RAG

회사 내부 RAG는 보통 아래 같은 형태로 쓰입니다.

- 사내 위키 검색
- 보안 정책 문서 질의응답
- 인사 규정, 복지, 결재 절차 안내
- 고객 대응용 내부 운영 매뉴얼 검색
- 개발자용 내부 API 문서 검색

### 왜 보안형 RAG가 필요한가

- 원문 문서가 외부 공개되면 안 된다.
- 부서별로 접근 가능한 문서가 다르다.
- 로그와 질문 자체에도 민감 정보가 포함될 수 있다.
- 문서 버전이 바뀌면 검색 결과도 빠르게 반영되어야 한다.

### 보안형 RAG에서 자주 붙는 요소

- VNET 또는 사설 연결
- 사용자 인증과 권한 기반 검색
- 문서별 접근 제어
- 입력/출력 콘텐츠 필터링
- 감사 로그와 모니터링

결국 보안형 RAG의 핵심은, `누가 어떤 문서를 어떤 경로로 보고 있는가`를 통제하는 것입니다.


## 청킹이란 무엇인가

청킹은 긴 문서를 검색 가능한 작은 단위인 `chunk`로 나누는 과정입니다.

왜 필요할까요?

- 문서 전체를 한 번에 임베딩하면 너무 크거나 비효율적일 수 있습니다.
- 검색은 질문과 가장 관련 있는 부분을 찾아야 하기 때문에, 적절한 단위로 쪼개야 합니다.
- 한 청크가 너무 크면 잡음이 많아지고, 너무 작으면 문맥이 끊깁니다.

즉, 청킹은 단순히 텍스트를 자르는 작업이 아니라 **검색 품질을 설계하는 작업**입니다.


## 청킹 Flow

청킹은 보통 아래 순서로 생각하면 좋습니다.

1. 원문 문서 구조 파악
2. 정제 대상 확인
3. 어떤 경계를 우선 보존할지 결정
4. 청크 크기와 overlap 결정
5. 청크 생성
6. 메타데이터 부여
7. 검색 결과를 보고 다시 조정

예를 들어 문서 종류에 따라 우선 보존해야 하는 경계가 달라집니다.

- 정책 문서: 장/절/항 경계
- 기술 문서: 제목/문단/코드 블록 경계
- FAQ: 질문-답변 쌍 경계
- 표가 많은 문서: 표 전체 경계


## 청킹 유형 1: CharacterTextSplitter

문자 수 기준으로 텍스트를 자르는 가장 단순한 방식입니다.

### 장점

- 이해하기 쉽습니다.
- 빠르게 실험하기 좋습니다.
- 문서 구조가 단순할 때 시작점으로 괜찮습니다.

### 단점

- 문장이나 문단 경계를 잘 보존하지 못할 수 있습니다.
- 의미 단위가 중간에 잘릴 위험이 큽니다.

### 잘 맞는 경우

- 아주 짧고 단순한 텍스트
- 빠른 프로토타이핑


## 청킹 유형 2: RecursiveCharacterTextSplitter

보통 가장 많이 쓰는 기본형입니다. 큰 경계에서 먼저 나누고, 안 되면 더 작은 경계로 내려가며 나눕니다.

예를 들어 아래 같은 순서로 경계를 시도할 수 있습니다.

```text
문단 -> 줄바꿈 -> 문장 -> 공백 -> 문자
```

### 장점

- 단순 character split보다 문맥 보존에 유리합니다.
- 대부분의 일반 문서에서 출발점으로 좋습니다.
- 문단 구조가 있는 문서에 특히 무난합니다.

### 단점

- 토큰 수를 정확히 제어하는 것은 아닙니다.
- 언어와 문서 형식에 따라 경계 설정을 조정해야 할 수 있습니다.

### 잘 맞는 경우

- 일반 문서, 사내 위키, 설명형 문서
- 처음 청킹 실험을 시작할 때


## 청킹 유형 3: TokenTextSplitter

모델이 실제로 처리하는 `토큰` 단위에 더 가깝게 나누는 방식입니다.

### 장점

- 모델 컨텍스트 길이를 기준으로 더 직접적으로 관리할 수 있습니다.
- 토큰 예산이 중요한 시스템에서 유용합니다.

### 단점

- 사람이 보기에 자연스러운 문장 경계와 항상 맞지 않을 수 있습니다.
- 언어별 토큰화 특성을 이해하지 않으면 결과가 어색할 수 있습니다.

### 잘 맞는 경우

- 모델 입력 길이를 엄격히 관리해야 하는 경우
- 긴 문서를 다룰 때 토큰 기준 실험이 필요한 경우


In [ ]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

sample_text = """
RAG는 검색과 생성을 결합하는 접근입니다.
회사 내부 문서를 다룰 때는 보안, 권한, 네트워크 격리도 함께 고려해야 합니다.
청킹이 너무 거칠면 검색 정확도가 떨어지고, 너무 잘게 나누면 문맥이 끊깁니다.
""".strip()


In [ ]:
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=40,
    chunk_overlap=10,
)

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=60,
    chunk_overlap=15,
)

token_splitter = TokenTextSplitter(
    chunk_size=30,
    chunk_overlap=5,
)

print("Character:", char_splitter.split_text(sample_text))
print("Recursive:", recursive_splitter.split_text(sample_text))
print("Token:", token_splitter.split_text(sample_text))


## 적절한 청킹 방법은 어떻게 고를까

정답은 하나가 아니라, **문서 형식과 질문 패턴에 맞는 선택**이 중요합니다.

### 시작점으로 많이 쓰는 전략

- 일반 문서: `RecursiveCharacterTextSplitter`
- 엄격한 입력 길이 관리가 필요할 때: `TokenTextSplitter`
- 아주 단순한 실험: `CharacterTextSplitter`

### 크기와 overlap에 대한 감각

- 청크가 너무 작으면: 검색은 잘 되더라도 의미가 잘려 답변 품질이 낮아질 수 있음
- 청크가 너무 크면: 관련 없는 정보가 많이 섞여 검색 정밀도가 낮아질 수 있음
- overlap이 너무 작으면: 경계에서 문맥 손실이 큼
- overlap이 너무 크면: 중복이 늘고 검색 결과가 비슷한 청크로 도배될 수 있음

### 초심자용 기본 출발점 예시

- 설명형 문서: `400~800 토큰`, `50~120 토큰 overlap`
- 짧은 FAQ: 더 작은 청크도 가능
- 표, 코드, 규정 문서: 구조 단위를 먼저 보존하고 그다음 크기 조정


## 문맥 손실, 청크 경계 처리, 그리고 흔한 문제

청킹에서 가장 흔한 실패는 **의미 단위가 중간에 잘리는 것**입니다.

예를 들어 아래 같은 문제가 생길 수 있습니다.

- 질문은 정책의 예외 조항을 묻는데, 예외 조항이 앞 청크와 뒤 청크에 나뉘어 들어감
- 코드 설명과 코드 블록이 서로 다른 청크로 갈라짐
- 표 제목과 표 본문이 분리됨

### 경계 처리 전략

- overlap을 적절히 둔다.
- 제목, 문단, 표, 코드 블록처럼 구조적 경계를 우선 보존한다.
- 메타데이터에 문서명, 섹션명, 페이지 번호를 넣는다.
- 문장 단위 또는 의미 단위 분할을 우선 고려한다.
- 검색 결과를 실제 질문으로 테스트하면서 조정한다.

중요한 점은, 청킹 품질은 코드 한 줄로 끝나는 문제가 아니라 **문서 구조 이해 + 검색 결과 평가**의 문제라는 점입니다.


## RAG 초심자가 자주 헷갈리는 포인트

### 1. RAG는 학습이 아니다

문서를 넣는다고 모델이 그 내용을 영구히 기억하는 것은 아닙니다. 질문 시점에 관련 문서를 가져와 붙이는 것입니다.

### 2. 파인튜닝은 지식 검색의 대체재가 아니다

최신 사내 문서처럼 자주 바뀌는 정보는 파인튜닝보다 RAG가 더 적합한 경우가 많습니다.

### 3. 벡터 DB가 있다고 자동으로 잘 찾는 것은 아니다

문서 정제, 청킹, 임베딩, 메타데이터, 질의 방식이 모두 품질에 영향을 줍니다.

### 4. 청킹은 작을수록 좋은 것이 아니다

너무 작은 청크는 검색은 되더라도 답변에 필요한 문맥이 부족할 수 있습니다.

### 5. 보안형 RAG는 검색 정확도만으로 평가할 수 없다

누가 어떤 문서에 접근했는지, 로그가 안전한지, 필터링이 적절한지도 같이 봐야 합니다.


## 1주차 체크리스트

아래 질문에 스스로 답할 수 있으면 1주차 기초가 잘 잡힌 상태입니다.

- RAG와 파인튜닝의 차이를 한 문장씩 설명할 수 있는가
- 회사 내부 RAG에서 왜 보안과 권한 관리가 중요한지 설명할 수 있는가
- RAG 파이프라인을 적재 단계와 질의 단계로 나눠 말할 수 있는가
- 청킹이 retrieval 품질에 왜 중요한지 설명할 수 있는가
- Character, Recursive, Token split의 차이를 말할 수 있는가
- overlap이 필요한 이유를 예시와 함께 설명할 수 있는가


## 스스로 점검해 볼 질문

1. 사내 보안 정책 문서를 검색하는 챗봇을 만든다면, RAG와 파인튜닝 중 무엇이 우선일까
2. FAQ 문서와 긴 정책 문서는 같은 방식으로 청킹해도 될까
3. 보안형 RAG에서 콘텐츠 필터링은 입력과 출력 중 어디에 필요한가
4. 검색은 잘 되는데 답변이 자꾸 엉뚱하면, 청킹과 프롬프트 중 어디를 먼저 의심할까
5. 사내 문서가 매주 바뀌는 환경에서 파인튜닝만으로 대응하기 어려운 이유는 무엇일까


## 다음 공부로 연결하기

1주차에서 중요한 것은 용어를 외우는 것보다 **흐름을 이해하는 것**입니다.

다음 단계에서는 아래 순서로 넘어가면 좋습니다.

- 2주차: chunk size, overlap, split 방식 비교 실습
- 3주차: 임베딩과 벡터 저장소 연결
- 4주차: 문서 적재부터 검색까지 기본 RAG 파이프라인 구축

이 노트북을 다 읽었다면, 이제 실제 문서를 가지고 `청킹을 바꿨을 때 검색 결과가 어떻게 달라지는지` 실습해 보는 것이 가장 좋습니다.
